# 🚀 Vibecode Lab — Generador de Video e Imágenes con IA a Costo $0
### **Laboratorio Práctico: Inferencia Multimedia en Cloud con GPU NVIDIA T4 (15 GB VRAM)**

Este notebook es parte del material práctico de **Vibecode Lab (Segunda Generación)**.

**¿Por qué usar Google Colab en vez de tu computador local?**
- ✅ **Costo $0:** Google Colab te da acceso gratuito a una GPU **NVIDIA Tesla T4 de 15 GB VRAM**.
- ✅ **Cero desgaste de hardware:** No necesitas una tarjeta gráfica gamer ni calentar tu laptop.
- ✅ **Multiplataforma:** Funciona igual desde Mac, Windows, Linux o incluso una tablet.
- ✅ **Sin infierno de dependencias:** Los drivers CUDA y PyTorch vienen listos para usar.

## ⚙️ Paso 0: Verificar la GPU Gratuita Asignada
Asegúrate de ir a **Entorno de ejecución > Cambiar tipo de entorno de ejecución** y seleccionar **T4 GPU**.
Al ejecutar la siguiente celda, verás tu GPU NVIDIA con ~15.000 MiB de VRAM.

In [ ]:
# Verificamos la GPU NVIDIA disponible en el entorno
!nvidia-smi

## 📦 Paso 1: Instalación de Dependencias de Inferencia
Instalamos la suite de Diffusers de HuggingFace optimizada con `accelerate` e `imageio[ffmpeg]` para compresión de video.

In [ ]:
!pip install -q diffusers transformers accelerate imageio[ffmpeg] safetensors torch

## 🎬 Paso 2: Cargar el Pipeline de Video (AnimateDiff)
Usamos **AnimateDiff** con el adaptador de movimiento v1.5-2 y un checkpoint realista para generar secuencias de video a partir de texto.

In [ ]:
import torch
from diffusers import AnimateDiffPipeline, MotionAdapter, DDIMScheduler
from diffusers.utils import export_to_video

print("📥 Cargando Adaptador de Movimiento AnimateDiff...")
adapter = MotionAdapter.from_pretrained(
    "guoyww/animatediff-motion-adapter-v1-5-2",
    torch_dtype=torch.float16
)

print("📥 Cargando Pipeline base con Stable Diffusion...")
pipe = AnimateDiffPipeline.from_pretrained(
    "emilianJR/epiCRealism",
    motion_adapter=adapter,
    torch_dtype=torch.float16
)

# Optimizamos el scheduler para transiciones fluidas
pipe.scheduler = DDIMScheduler.from_config(
    pipe.scheduler.config,
    clip_sample=False,
    timestep_spacing="linspace",
    beta_schedule="linear",
    steps_offset=1
)

# Enviamos el pipeline a la GPU NVIDIA T4
pipe.to("cuda")
# pipe.enable_vae_slicing() no es necesario en GPU T4 de 15GB
print("✅ Pipeline de Video listo en GPU T4 (15GB)!")

## 🎥 Paso 3: Generar tu Primer Video con IA
Define tu prompt en inglés para máxima calidad. El pipeline creará 16 fotogramas con movimiento cinemático.

In [ ]:
# Configura tu prompt
prompt = "cinematic shot of a futuristic sleek humanoid robot in a neon green server room, smooth head motion, hyperrealistic, 8k, photorealistic, intricate circuitry"
negative_prompt = "blurry, low quality, distorted, deformed, static, bad anatomy, cartoon"

print(f"🎬 Generando video para prompt: {prompt}")
output = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_frames=16,
    guidance_scale=7.5,
    num_inference_steps=25,
    generator=torch.Generator("cuda").manual_seed(42)
)

frames = output.frames[0]
video_path = "vibecode_generated_video.mp4"
export_to_video(frames, video_path, fps=8)
print(f"✅ Video generado y guardado exitosamente en: {video_path}")

## 📺 Paso 4: Visualizar y Descargar el Video
Ejecuta esta celda para ver el video generado directamente en el notebook y descargarlo a tu equipo.

In [ ]:
from IPython.display import Video, display
from google.colab import files

# Mostramos el video en el notebook
display(Video(video_path, embed=True, width=512, height=512))

# Descomenta la siguiente línea si deseas descargarlo automáticamente a tu PC:
# files.download(video_path)

## ⚡ Paso 5 (Bonus): Generación de Imágenes Ultrarrápidas con SDXL Lightning
Con la GPU T4 de 15 GB, también puedes generar imágenes hiperrealistas en **menos de 2 segundos** usando SDXL Lightning de 4 pasos.

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline, UNet2DConditionModel, EulerDiscreteScheduler
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

base = "stabilityai/stable-diffusion-xl-base-1.0"
repo = "ByteDance/SDXL-Lightning"
ckpt = "sdxl_lightning_4step_unet.safetensors"

print("📥 Cargando SDXL Lightning...")
unet = UNet2DConditionModel.from_config(base, subfolder="unet").to("cuda", torch.float16)
unet.load_state_dict(load_file(hf_hub_download(repo, ckpt), device="cuda"))
pipe_img = StableDiffusionXLPipeline.from_pretrained(base, unet=unet, torch_dtype=torch.float16, variant="fp16").to("cuda")

# Configuración de scheduler de 4 pasos
pipe_img.scheduler = EulerDiscreteScheduler.from_config(pipe_img.scheduler.config, timestep_spacing="trailing")

img_prompt = "A sleek futuristic SaaS dashboard floating in dark space, neon emerald and purple neural network lines, glassmorphism, 8k resolution, ultra detailed"
image = pipe_img(img_prompt, num_inference_steps=4, guidance_scale=0).images[0]

image.save("vibecode_saas_preview.png")
display(image)
print("✅ Imagen de alta resolución generada en 4 pasos!")

## 🎓 Conclusión & Siguiente Nivel en Vibecode Lab
En **Vibecode Lab**, aprenderás a:
1. Envolver estos pipelines en **APIs serverless** (con Modal, HuggingFace Inference Endpoints o RunPod).
2. Conectarlos a tu frontend en **Next.js y Tailwind CSS**.
3. Cobrar por cada generación conectando **Mercado Pago** y guardando los resultados en **Supabase Storage**.

👉 Visita [https://consultoria-v0-jm.vercel.app/vibecode](https://consultoria-v0-jm.vercel.app/vibecode) para más información del bootcamp.